# Qwen3-4B Fine-tuning with Unsloth on Google Colab

Fine-tune Qwen3-4B on your custom dataset with advanced memory optimizations.

**Features:**
- 🦥 Unsloth Dynamic 2.0 (2x faster training)
- 💾 Embedding offload (saves 1GB VRAM)
- 🚀 Experiment mode for fast testing
- 📊 Train/eval split monitoring
- 🧪 Inference testing with both thinking modes

**Dataset:** Upload your `training_codex.jsonl` file to Colab

## Step 1: Setup & Installation

Install Unsloth and dependencies (takes ~3 minutes)

In [ ]:
%%capture
import os, re

# Install Unsloth for Colab
if "COLAB_" in "".join(os.environ.keys()):
    import torch
    v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
    !pip install transformers==4.56.2
    !pip install --no-deps trl==0.22.2
else:
    !pip install unsloth

print("✅ Installation complete!")

## Step 2: Upload Your Dataset

**Upload your `training_codex.jsonl` file using the file icon on the left sidebar** 📁

Or run this cell to use the file upload button:

In [ ]:
from google.colab import files
import os

print("📤 Upload your training_codex.jsonl file:")
uploaded = files.upload()

# Get the uploaded filename
dataset_file = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {dataset_file}")
print(f"   Size: {os.path.getsize(dataset_file) / 1024:.2f} KB")

# Verify it's a JSONL file
if dataset_file.endswith('.jsonl'):
    import json
    with open(dataset_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"   Lines: {len(lines)}")
    
    # Verify format
    try:
        first_example = json.loads(lines[0])
        if "conversations" in first_example:
            print(f"   Format: ✅ Valid JSONL with 'conversations' field")
        else:
            print(f"   ⚠️  Warning: No 'conversations' field found")
    except:
        print(f"   ⚠️  Warning: Could not parse as JSON")
else:
    print(f"   ⚠️  Warning: File doesn't end with .jsonl")

## Step 3: Configuration

Adjust these settings based on your needs:

In [ ]:
# =====================================================================
# CONFIGURATION - Modify as needed
# =====================================================================

# Model Settings
MODEL_NAME = "unsloth/Qwen3-4B"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
OFFLOAD_EMBEDDING = True  # Saves 1GB VRAM

# Training Mode
EXPERIMENT_MODE = True  # Set True for fast testing (r=4), False for production (r=32)
LORA_R = 4 if EXPERIMENT_MODE else 32
LORA_ALPHA = LORA_R * 2  # Faster convergence

# Training Settings
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
MAX_STEPS = 60  # For testing; set to None and uncomment num_train_epochs for full training
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.1

# Evaluation
ENABLE_EVAL = True  # Monitor overfitting
EVAL_SPLIT_SIZE = 0.05  # 5% for validation
EVAL_STEPS = 10

# Memory Management
AGGRESSIVE_MEMORY_CLEANUP = True

# Dataset File (from upload)
DATASET_PATH = dataset_file  # Automatically set from upload

print("📋 Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Mode: {'🧪 Experiment (r=4, fast)' if EXPERIMENT_MODE else '🏭 Production (r=32)'}")
print(f"   LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Max Steps: {MAX_STEPS}")
print(f"   Evaluation: {'✅ Enabled' if ENABLE_EVAL else '❌ Disabled'}")
print(f"   Dataset: {DATASET_PATH}")

## Step 4: Load Model with Memory Optimization

In [ ]:
from unsloth import FastLanguageModel
import torch
import gc

print("🚀 Loading Qwen3-4B with optimizations...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    offload_embedding = OFFLOAD_EMBEDDING,  # Saves ~1GB VRAM
)

# Initial memory cleanup
gc.collect()
torch.cuda.empty_cache()

print("✅ Model loaded!")
print(f"   Embedding offload: {'✅ Active (saves ~1GB)' if OFFLOAD_EMBEDDING else '❌ Disabled'}")

## Step 5: Apply LoRA Adapters

In [ ]:
print(f"🔧 Applying LoRA adapters (r={LORA_R}, alpha={LORA_ALPHA})...")

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state = 3407,
)

print("✅ LoRA adapters applied!")

## Step 6: Prepare Dataset

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

print("📊 Preparing dataset...")

# Get Qwen3 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-thinking",
)

# Load your uploaded dataset
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print(f"   Loaded {len(dataset)} examples")

# Format with chat template
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, 
            tokenize=False, 
            add_generation_prompt=False,
            enable_thinking=False  # Set based on your dataset
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# Split into train/eval if enabled
if ENABLE_EVAL and EVAL_SPLIT_SIZE > 0:
    dataset = dataset.train_test_split(test_size=EVAL_SPLIT_SIZE, seed=3407)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]
    print(f"   📚 Train: {len(train_dataset)} examples")
    print(f"   📊 Eval: {len(eval_dataset)} examples")
else:
    train_dataset = dataset
    eval_dataset = None
    print(f"   📚 Training on all {len(train_dataset)} examples")

# Memory cleanup
gc.collect()
torch.cuda.empty_cache()

print("✅ Dataset prepared!")

## Step 7: Configure Trainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

print("🏋️ Configuring trainer...")

trainer_args = SFTConfig(
    dataset_text_field = "text",
    per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION,
    warmup_ratio = WARMUP_RATIO,
    max_steps = MAX_STEPS,
    # num_train_epochs = 1,  # Uncomment for full training
    learning_rate = LEARNING_RATE,
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    report_to = "none",
)

# Add evaluation if enabled
if ENABLE_EVAL and eval_dataset is not None:
    trainer_args.eval_strategy = "steps"
    trainer_args.eval_steps = EVAL_STEPS
    trainer_args.per_device_eval_batch_size = 4
    trainer_args.eval_accumulation_steps = 1

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = trainer_args,
)

# Train only on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

print("✅ Trainer configured!")

## Step 8: Add Memory Cleanup Callback (Optional but Recommended)

In [ ]:
from transformers.trainer_callback import TrainerCallback

if AGGRESSIVE_MEMORY_CLEANUP:
    class MemoryCleanupCallback(TrainerCallback):
        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step % 10 == 0:  # Every 10 steps
                gc.collect()
                torch.cuda.empty_cache()
    
    trainer.add_callback(MemoryCleanupCallback())
    print("✅ Memory cleanup callback enabled (runs every 10 steps)")

## Step 9: Train! 🚀

This will show training progress. Watch the loss decrease!

In [ ]:
print("="*70)
print("🚀 STARTING TRAINING")
print("="*70)

# Show GPU stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Memory reserved: {start_gpu_memory} GB")
print("="*70 + "\n")

# Train!
trainer_stats = trainer.train()

# Post-training stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"⏱️  Time: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes")
print(f"💾 Peak memory: {used_memory} GB ({used_percentage}% of {max_memory} GB)")
print(f"📊 Training memory: {used_memory_for_lora} GB")
print("="*70)

# Final cleanup
gc.collect()
torch.cuda.empty_cache()

## Step 10: Save Model

In [ ]:
print("💾 Saving LoRA adapters...")

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

print("✅ Model saved to ./lora_model/")
print("\n📦 You can download this folder to use locally!")

## Step 11: Test Inference

Try both thinking and non-thinking modes!

In [ ]:
from transformers.generation.streamers import TextStreamer

# Enable fast inference
FastLanguageModel.for_inference(model)

# Inference settings (Qwen3 official recommendations)
NON_THINKING_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0.0,
    "max_new_tokens": 512,
}

THINKING_PARAMS = {
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.0,
    "max_new_tokens": 1024,
}

print("="*70)
print("🧪 INFERENCE TEST 1: Non-Thinking Mode")
print("="*70)
print("Question: What is Null Singularity (Ω_∅)?")
print("-"*70 + "\n")

messages = [{"role": "user", "content": "What is Null Singularity (Ω_∅)?"}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    **NON_THINKING_PARAMS,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

# Cleanup between generations
gc.collect()
torch.cuda.empty_cache()

print("\n\n" + "="*70)
print("🧪 INFERENCE TEST 2: Thinking Mode")
print("="*70)
print("Question: Why can't I 'grasp' Null Singularity?")
print("-"*70 + "\n")

messages = [{"role": "user", "content": "Why can't I 'grasp' Null Singularity (Ω_∅)?"}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    **THINKING_PARAMS,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

print("\n" + "="*70)
print("✅ INFERENCE TESTS COMPLETE!")
print("="*70)

## Step 12: Try Your Own Questions!

Modify the question below and run the cell:

In [ ]:
# 🎯 Try your own question here!
YOUR_QUESTION = "What is the relationship between form and freedom?"

print("="*70)
print(f"❓ Your Question: {YOUR_QUESTION}")
print("="*70 + "\n")

messages = [{"role": "user", "content": YOUR_QUESTION}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False  # Change to True for thinking mode
)

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    **NON_THINKING_PARAMS,  # Or use THINKING_PARAMS
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

## Step 13: Download Your Fine-tuned Model

Download the LoRA adapters to use locally or export to other formats.

In [ ]:
from google.colab import files
import shutil

print("📦 Preparing model for download...")

# Create a zip file
shutil.make_archive('lora_model', 'zip', 'lora_model')

print("⬇️ Downloading lora_model.zip...")
files.download('lora_model.zip')

print("\n✅ Download complete!")
print("\n📝 To use locally:")
print("   1. Unzip lora_model.zip")
print("   2. Load with: FastLanguageModel.from_pretrained('lora_model')")

## Optional: Export to GGUF (for Ollama/llama.cpp)

Uncomment and run if you want GGUF format:

In [ ]:
# Uncomment to export to GGUF format:

# print("📦 Exporting to GGUF (Q4_K_M format)...")
# model.save_pretrained_gguf("model", tokenizer, quantization_method="q4_k_m")
# print("✅ GGUF saved to ./model/")

# # Download GGUF
# !zip -r model_gguf.zip model/
# files.download('model_gguf.zip')

## 🎉 Training Complete!

### What You've Done:
✅ Fine-tuned Qwen3-4B on your custom dataset  
✅ Used Unsloth Dynamic 2.0 for 2x faster training  
✅ Applied memory optimizations (embedding offload, cleanup)  
✅ Tested inference with both thinking modes  
✅ Downloaded your fine-tuned model  

### Next Steps:
1. **Test more questions** using the cell above
2. **Export to GGUF** for Ollama/llama.cpp deployment
3. **Share on Hugging Face** (optional)
4. **Use in production** with your philosophical framework

### Resources:
- [Unsloth Docs](https://docs.unsloth.ai/)
- [Qwen3 Guide](https://docs.unsloth.ai/basics/qwen3-how-to-run-and-fine-tune)
- [Unsloth Discord](https://discord.gg/unsloth)

---

**Trained with** 🦥 Unsloth | **Optimized for** Qwen3-4B | **Ready for** deployment